# Hierarchical simulator: groups of clients

Extends `simulator.ipynb` for approaches where clients work in **groups**: each group trains **synchronously** (a group round waits for all its selected clients), and the groups update the global model **asynchronously**. Examples: FedAT (tiers of clients with similar speed) and FedCure (coalitions under edge servers).

**How it's used:** a hierarchical approach notebook starts with

```python
%run ../hierarchical_simulator.ipynb
```

That gives it everything from `simulator.ipynb` (settings, data, devices, models, clients, the event loop, `run()`, saving), plus the two additions below. Results are saved exactly like the other approaches, so `compare.ipynb` and `run_all.ipynb` include them automatically.

> **Important:** this notebook does **not** change `simulator.ipynb`. It loads it and adds to it. Approaches that use `simulator.ipynb` directly are not affected.

| Section | What it does |
|---|---|
| 1. Load the shared simulator | everything from `simulator.ipynb` |
| 2. Hierarchical server | the one new ability: start a client from a model other than the global one |
| **3. Group approach API** | **the only section you need to read to write a group-based approach** |
| 4. Helpers | `average_models` |

## 1. Load the shared simulator

The code cells of `simulator.ipynb` are run **in this notebook's namespace** (not with a nested `%run`), so that `run()` builds the hierarchical server defined below.

> **Note:** `simulator.ipynb` is found in this folder or a parent folder, so approach notebooks in `approaches/` work too.

In [ ]:
import math
from pathlib import Path
import nbformat

_folder = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "simulator.ipynb").exists())
for _cell in nbformat.read(_folder / "simulator.ipynb", as_version=4).cells:
    if _cell.cell_type == "code":
        exec(_cell.source)

## 2. Hierarchical server

`HierarchicalServer` is the normal `Server` with one extra ability: `server.train(client_id, weights=...)` starts a client from the given weights (for example an edge or tier model) instead of the global model. Without `weights` it behaves exactly like `Server`.

> **Note:** `run()` builds a `HierarchicalServer` from here on (in notebooks that load this one). Flat approaches never pass `weights`, so they give identical results with either server.

In [ ]:
class HierarchicalServer(Server):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.start_weights = {}                    # client id -> (weights its next job starts from, global version then)

    def train(self, client_id, at=None, weights=None):
        if weights is not None:
            self.start_weights[client_id] = (weights, self.version)
        started = super().train(client_id, at)
        if not started:
            self.start_weights.pop(client_id, None)
        return started

    def _begin(self, c):
        given = self.start_weights.pop(c.id, None)
        if given is None:
            return super()._begin(c)
        weights, version = given                  # staleness counts from when the weights were given, not from download
        c.begin(weights, version, self.approach.local_epochs(self, c), self.S.lr, self.approach.local_loss)


Server = HierarchicalServer      # run() builds this server

## 3. Group approach API

**This is the only section you need to read to write a group-based approach.** Subclass `GroupApproach` and override what you need. The base class does the bookkeeping: it starts a group round, waits until every selected client of the group has reported, averages them into the group model, repeats this `group_rounds` times, and then hands the group model to `on_group_done`.

| Method | Called when | Default | Override it for |
|---|---|---|---|
| `form_groups(server)` | t = 0 | one group with all clients | tiers, coalitions |
| `start_model(server, g)` | group `g` starts a job | the global model | edge or tier models |
| `pick_members(server, g)` | every group round | all clients of the group | sampling some clients |
| `aggregate(server, g, updates)` | a group round is complete | FedAvg of the clients' models | other intra-group rules |
| **`on_group_done(server, g, model, updates)`** | **group `g` finished its `group_rounds`** | **must be written** | **the global (cloud) update** |
| `next_groups(server, g)` | after `on_group_done` | `[g]`: the same group starts again | scheduling |
| `first_groups(server)` | t = 0 | every group | starting only some groups |

`local_loss` and `local_epochs` work as in `simulator.ipynb` (section 6).

**What you can use in a group approach:** everything on `server` from `simulator.ipynb`, plus `self.groups` (list of client-id lists), `self.group_of[client_id]`, `self.group_model[g]`, and `self.group_staleness(server, g)` (global versions since group `g` received the global model).

> **Tip:** `group_rounds = 1` means every group round goes to the cloud (FedAT). A larger value means several edge rounds before the cloud is updated (FedCure).

> **Note:** if the update budget (`max_updates`) runs out between two group rounds, the group reports the rounds it has finished, so no client training is wasted.

In [ ]:
class GroupApproach(Approach):
    group_rounds = 1               # synchronous group rounds before the group reports to the cloud

    # ---------------- override these ----------------
    def form_groups(self, server):
        return [[c.id for c in server.clients]]

    def start_model(self, server, g):
        return server.model

    def pick_members(self, server, g):
        return list(self.groups[g])

    def aggregate(self, server, g, updates):
        return fedavg(updates)

    def on_group_done(self, server, g, model, updates):
        raise NotImplementedError("on_group_done must be written")

    def next_groups(self, server, g):
        return [g]

    def first_groups(self, server):
        return range(len(self.groups))

    # ---------------- bookkeeping (no need to change) ----------------
    def group_staleness(self, server, g):
        return server.version - self.group_version[g]

    def start(self, server):
        self.groups = [list(members) for members in self.form_groups(server)]
        self.group_of = {cid: g for g, members in enumerate(self.groups) for cid in members}
        self.group_model, self.group_version = {}, {}
        self.round, self.expected, self.received, self.job_updates = {}, {}, {}, {}
        self.active = set()
        for g in self.first_groups(server):
            self.start_group(server, g)

    def start_group(self, server, g):
        """Group g starts a job: group_rounds synchronous rounds, then on_group_done."""
        self.group_version[g] = server.version
        self.group_model[g] = self.start_model(server, g)
        self.round[g], self.job_updates[g] = 0, []
        self._start_round(server, g)

    def _start_round(self, server, g):
        started = [cid for cid in self.pick_members(server, g) if server.train(cid, weights=self.group_model[g])]
        self.expected[g], self.received[g] = len(started), []
        if started:
            self.active.add(g)
        else:
            self.active.discard(g)                 # update budget used up

    def client_free(self, server, client):
        pass                                       # the group decides when its clients train again

    def on_update(self, server, update):
        g = self.group_of[update.client_id]
        self.received[g].append(update)
        if len(self.received[g]) < self.expected[g]:
            return                                 # wait for the rest of the group
        updates = sorted(self.received[g], key=lambda u: u.client_id)
        self.job_updates[g] += updates
        self.group_model[g] = self.aggregate(server, g, updates)
        self.round[g] += 1
        if self.round[g] < self.group_rounds:
            self._start_round(server, g)           # next synchronous round inside the group
            if g in self.active:
                return
            # the update budget is used up: report the rounds finished so far instead of dropping them
        self.active.discard(g)
        self.on_group_done(server, g, self.group_model[g], self.job_updates[g])
        for nxt in self.next_groups(server, g):
            if nxt not in self.active:
                self.start_group(server, nxt)

## 4. Helpers

* `average_models(models, weights)`: the weighted average of full models (state dicts). Weights are normalised to sum to 1.

In [ ]:
def average_models(models, weights):
    total = float(sum(weights))
    out = {}
    for k, ref in models[0].items():
        if not torch.is_floating_point(ref):
            out[k] = ref.clone()
            continue
        out[k] = sum(m[k] * (w / total) for m, w in zip(models, weights))
    return out